In [2]:
import numpy as np
# Patch for NumPy >= 1.24 where np.bool8 was removed
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_
if not hasattr(np, "np.float_"):
    np.float_ = np.float64
if not hasattr(np, "np.complex_"):
    np.complex_ = np.complex128
if not hasattr(np, "np.unicode_"):
    np.unicode_ = np.str_

import torch
# Quick patch for PyTorch 2.6+ to fix UnpicklingError
orig_torch_load = torch.load
torch.load = lambda f, *args, **kwargs: orig_torch_load(f, *args, weights_only=False, **kwargs)

In [3]:
from recbole.quick_start import run_recbole
run_recbole(model='NeuMF', dataset='ml-100k')

29 Sep 18:36    INFO  ['c:\\Users\\martin\\miniconda3\\envs\\bp\\Lib\\site-packages\\ipykernel_launcher.py', '--f=c:\\Users\\martin\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3e87f340f802189bce0f579f11ed99743714a4951.json']
29 Sep 18:36    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = C:\Users\martin\miniconda3\envs\bp\Lib\site-packages\recbole\config\../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 4

Evaluation Hyper Parameters:
eval_args = {'split': {'RS': [0.

{'best_valid_score': np.float64(0.3764),
 'valid_score_bigger': True,
 'best_valid_result': OrderedDict([('recall@10', np.float64(0.2143)),
              ('mrr@10', np.float64(0.3764)),
              ('ndcg@10', np.float64(0.2266)),
              ('hit@10', np.float64(0.7497)),
              ('precision@10', np.float64(0.1578))]),
 'test_result': OrderedDict([('recall@10', np.float64(0.2371)),
              ('mrr@10', np.float64(0.4618)),
              ('ndcg@10', np.float64(0.2795)),
              ('hit@10', np.float64(0.772)),
              ('precision@10', np.float64(0.1925))])}

In [ ]:
from recbole.quick_start import run_recbole
run_recbole(model='NeuMF', dataset='ml-100k')

In [ ]:
from recbole.quick_start import run_recbole

config_dict = {
    'data_path': './data/clean', # current folder
    'dataset': 'CustomDataset',   # folder + file prefix
    'user_inter_num_interval': "[0,100)",   # keep only users with < all items interactions 
    # "[a,b]" means keep users with a ≤ number of interactions ≤ b.
    # Common choices:
    # "[0,inf)" → drop only impossible users (those with all items).
    # "[5,inf)" → drop users with fewer than 5 interactions.
    "eval_setting": {
        "split": "leave_one_out",
        "group_by_user": True,
        "leave_one_num": 2,
        "order": "RO", 
    },
}

#run_recbole(model='NeuMF', dataset='CustomDataset', config_dict=config_dict)
run_recbole(model='NeuMF', config_dict=config_dict)

In [ ]:
import pandas as pd
import numpy as np

# --- config ---
input_path = "./data/clean/CustomDataset/CustomDataset.inter"   # your raw csv file
output_path = "./data/clean/CustomDataset/CustomDataset_split.inter"

random_seed = 42
np.random.seed(random_seed)

# --- load ---
df = pd.read_csv(input_path, sep="\t")

#df.head()

# ensure correct column names for RecBole
#df.columns = ['user_id:token', 'item_id:token', 'rating:float']

# --- remove users who interacted with all items ---
n_items = df['item_id:token'].nunique()
user_item_counts = df.groupby('user_id:token')['item_id:token'].nunique()

bad_users = user_item_counts[user_item_counts >= n_items].index.tolist()
print(f"Found {len(bad_users)} users who interacted with all {n_items} items → removing them.")

df = df[~df['user_id:token'].isin(bad_users)]

# --- add split column ---
df['split:token'] = 'train'  # default

def assign_splits(group):
    """Assign 1 val, 1 test, rest train for each user."""
    if len(group) >= 2:
        chosen = np.random.choice(group.index, size=min(2, len(group)), replace=False)
        group.loc[chosen[0], 'split:token'] = 'val'
        if len(chosen) > 1:
            group.loc[chosen[1], 'split:token'] = 'test'
    return group

df = df.groupby('user_id:token', group_keys=False).apply(assign_splits)

# --- save ---
df.to_csv(output_path, sep='\t', index=False)

print(f"Saved dataset with splits → {output_path}")

In [ ]:
from recbole.quick_start import run_recbole

config_dict = {
    'data_path': './data/clean', # current folder
    'dataset': 'test',   # folder + file prefix
    'eval_setting': {
        "split": "user_defined",
        "group_by_user": True
    },
    'epochs' : 10,
    'eval_step': 5, 
    'save_training': True,
    'neg_sampling': None,
    # 'train_batch_size': 1024,
    # 'eval_batch_size': 1024,
    'use_gpu': False,
    'num_workers': 1,
    'save_dataset': False,
    'save_model': False,
}

run_recbole(model='NeuMF', config_dict=config_dict)